In [ ]:
!pip install chromadb sentence-transformers -q

In [ ]:
print('Checking installation of chromadb:')
!pip show chromadb
print('\nChecking installation of sentence-transformers:')
!pip show sentence-transformers

Checking installation of chromadb:
Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: 

Checking installation of sentence-transformers:
Name: sentence-transformers
Version: 5.5.1
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, num

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb


In [ ]:
documents = [
    "Sleep in the whisper of sirens.",
    "How long will they do ot die",
    "A majestic feline predator, the lion, roams the savanna.",
    "jeeva is a naughty boy.",
    "the pigeons are flying. ",
    "Automobiles are a common mode of transportation."
]

print("Our document corpus:")
for i, doc in enumerate(documents):
    print(f"[{i}] {doc}")

Our document corpus:
[0] Sleep in the whisper of sirens.
[1] How long will they do ot die
[2] A majestic feline predator, the lion, roams the savanna.
[3] jeeva is a naughty boy.
[4] the pigeons are flying. 
[5] Automobiles are a common mode of transportation.


In [ ]:
def keyword_search(query, documents):
    results = []
    query_lower = query.lower()
    for i, doc in enumerate(documents):
        if query_lower in doc.lower():
            results.append((i, doc))
    return results

print("\n--- Keyword Search Results ---")
query_keyword = "dog"
keyword_results = keyword_search(query_keyword, documents)
print(f"Query: '{query_keyword}'")
if keyword_results:
    for idx, doc in keyword_results:
        print(f"Found in document [{idx}]: {doc}")
else:
    print("No exact keyword matches found.")

print("\n--- Keyword Search with synonym ---")
query_synonym = "canine"
keyword_results_synonym = keyword_search(query_synonym, documents)
print(f"Query: '{query_synonym}'")
if keyword_results_synonym:
    for idx, doc in keyword_results_synonym:
        print(f"Found in document [{idx}]: {doc}")
else:
    print("No exact keyword matches found.")


--- Keyword Search Results ---
Query: 'dog'
Found in document [0]: The quick brown fox jumps over the lazy dog.
Found in document [1]: Dogs are known as "man's best friend".

--- Keyword Search with synonym ---
Query: 'canine'
No exact keyword matches found.


In [ ]:
# Initialize the sentence-transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a ChromaDB client and collection
import chromadb
client = chromadb.Client()
collection = client.get_or_create_collection(name="my_documents")

# Generate embeddings for the documents
document_embeddings = model.encode(documents).tolist()

# Add documents and their embeddings to ChromaDB
collection.add(
    embeddings=document_embeddings,
    documents=documents,
    metadatas=[{"source": f"doc_{i}"} for i in range(len(documents))],
    ids=[f"doc_{i}" for i in range(len(documents))]
)

print("Embeddings generated and stored in ChromaDB.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings generated and stored in ChromaDB.


In [ ]:
def semantic_search(query, collection, model, n_results=2):
    # Generate embedding for the query
    query_embedding = model.encode([query]).tolist()

    # Query ChromaDB for similar documents
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )
    return results

print("\n--- Semantic Search Results ---")
query_semantic = "domestic pets"
semantic_results = semantic_search(query_semantic, collection, model, n_results=2)

print(f"Query: '{query_semantic}'")
if semantic_results and 'documents' in semantic_results and semantic_results['documents']:
    for i, doc_list in enumerate(semantic_results['documents']):
        for j, doc in enumerate(doc_list):
            distance = semantic_results['distances'][i][j]
            print(f"Relevant document (distance: {distance:.4f}): {doc}")
else:
    print("No semantic results found.")

print("\n--- Semantic Search with car synonym ---")
query_car_synonym = "vehicles for transport"
semantic_results_car = semantic_search(query_car_synonym, collection, model, n_results=1)

print(f"Query: '{query_car_synonym}'")
if semantic_results_car and 'documents' in semantic_results_car and semantic_results_car['documents']:
    for i, doc_list in enumerate(semantic_results_car['documents']):
        for j, doc in enumerate(doc_list):
            distance = semantic_results_car['distances'][i][j]
            print(f"Relevant document (distance: {distance:.4f}): {doc}")
else:
    print("No semantic results found.")


--- Semantic Search Results ---
Query: 'domestic pets'
Relevant document (distance: 0.9281): Dogs are known as "man's best friend".
Relevant document (distance: 1.1691): A majestic feline predator, the lion, roams the savanna.

--- Semantic Search with car synonym ---
Query: 'vehicles for transport'
Relevant document (distance: 0.6967): Automobiles are a common mode of transportation.


In [ ]:
# Define a single sentence to embed
single_sentence = "This is a test sentence for embedding."

# Embed the sentence
sentence_embedding = model.encode([single_sentence])

# Print the shape of the embedding (to show it's a vector)
print(f"Original sentence: '{single_sentence}'")
print(f"Shape of the embedding: {sentence_embedding.shape}")

# Print the first few dimensions of the embedding (or the whole vector if small enough)
print(f"First 5 dimensions of the embedding: {sentence_embedding[0][:5]}...")

Original sentence: 'This is a test sentence for embedding.'
Shape of the embedding: (1, 384)
First 5 dimensions of the embedding: [0.02782412 0.00170262 0.08005547 0.04666286 0.03852206]...


In [ ]:
# Perform a sample query using the first document as the query text
sample_query_text = five_docs[0]
query_results = collection.query(
    query_texts=[sample_query_text],
    n_results=1
)

# Display the keys of the result dictionary
print(f"Result keys: {query_results.keys()}")
print("\nFull result structure for reference:")
display(query_results)

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 82.2MiB/s]


Result keys: dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])

Full result structure for reference:


{'ids': [['doc_1']],
 'embeddings': None,
 'documents': [['Dogs are known as "man\'s best friend".']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'source': 'doc_1'}]],
 'distances': [[1.7988892793655396]]}

In [ ]:
print(f"--- Search Results for: '{sample_query_text}' ---\n")

# Extract lists from the results dictionary
ids = query_results['ids'][0]
documents = query_results['documents'][0]
metadatas = query_results['metadatas'][0]
distances = query_results['distances'][0]

# Iterate and display in a readable format
for i in range(len(ids)):
    print(f"Match #{i+1}")
    print(f"ID: {ids[i]}")
    print(f"Text: {documents[i]}")
    print(f"Score (Distance): {distances[i]:.4f}")
    print(f"Metadata: {metadatas[i]}")
    print("-" * 30)

--- Search Results for: 'Quantum computing leverages the principles of superposition and entanglement.' ---

Match #1
ID: doc_1
Text: Dogs are known as "man's best friend".
Score (Distance): 1.7989
Metadata: {'source': 'doc_1'}
------------------------------


In [ ]:
sample_docs = [
    "ETL stands for Extract, Transform and Load",
    "SQL SELECT statements retrieve data from database tables",
    "Machine learning models learn patterns from training data",
    "Python pandas library is used for data manipulation and cleaning",
    "Neural networks are inspired by how the human brain works",
]

sample_ids = ["doc001","doc002","doc003","doc004","doc005"]

sample_metadata = [
    {"subject": "Data engineering", "Topic": "ETL"},
    {"Subject": "Data engineering", "Topic": "SQL"},
    {"subject": "Machine learning", "Topic": "ML Basics"},
    {"subject": "Python", "Topic": "Pandas"},
    {"subject": "Machine learning", "Topic": "Neural networks"}
]

# Add the new documents to the existing collection
collection.add(
    documents=sample_docs,
    ids=sample_ids,
    metadatas=sample_metadata
)

print("Documents added to collection!")
print("Total Documents now in collection:", collection.count())

Documents added to collection!
Total Documents now in collection: 11


In [ ]:
filtered_query = "How do neural networks function?"

# Perform the filtered query
filtered_results = collection.query(
    query_texts=[filtered_query],
    n_results=3,
    where={"subject": "Machine learning"},
)

print("FILTERED QUERY:", filtered_query)
print("Filter: only machine learning documents")
print("="*60)

# Iterate through results using zip to handle multiple lists at once
for rank, (doc, dist, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['distances'][0],
    filtered_results['metadatas'][0]
), start=1):
    # Handle inconsistent casing for the 'subject' key
    subject_val = meta.get('subject') or meta.get('Subject')
    topic_val = meta.get('Topic')

    print(f"Rank: {rank} | Distance: {dist:.4f} | subject: {subject_val} | Topic: {topic_val}")
    print(doc)
    print()

FILTERED QUERY: How do neural networks function?
Filter: only machine learning documents
Rank: 1 | Distance: 0.6163 | subject: Machine learning | Topic: Neural networks
Neural networks are inspired by how the human brain works

Rank: 2 | Distance: 1.2935 | subject: Machine learning | Topic: ML Basics
Machine learning models learn patterns from training data



In [ ]:
search_query = "How do databases retrieve data using SQL?"

# Perform query
results = collection.query(
    query_texts=[search_query],
    n_results=3
)

print(f"--- Search Results for: '{search_query}' ---\n")

ids = results['ids'][0]
docs = results['documents'][0]
metas = results['metadatas'][0]
dists = results['distances'][0]

for i in range(len(ids)):
    print(f"Rank #{i+1} | ID: {ids[i]}")
    print(f"Text: {docs[i]}")
    print(f"Score (Distance): {dists[i]:.4f}")
    print(f"Metadata: {metas[i]}\n")

--- Search Results for: 'How do databases retrieve data using SQL?' ---

Rank #1 | ID: doc002
Text: SQL SELECT statements retrieve data from database tables
Score (Distance): 0.6220
Metadata: {'Topic': 'SQL', 'Subject': 'Data engineering'}

Rank #2 | ID: doc003
Text: Machine learning models learn patterns from training data
Score (Distance): 1.4776
Metadata: {'subject': 'Machine learning', 'Topic': 'ML Basics'}

Rank #3 | ID: doc004
Text: Python pandas library is used for data manipulation and cleaning
Score (Distance): 1.5926
Metadata: {'subject': 'Python', 'Topic': 'Pandas'}

